https://docs.langchain.com/oss/python/langchain/structured-output#tool-calling-strategy

ToolStrategy通过工具调用实现结构化输出，所以LangChain会在消息末尾追加一条ToolMessage，让整个链路完整。但实际上没有实际的工具执行，这是一条伪消息。

ToolStrategy适用任何支持工具调用的现代模型。

ToolStrategy的配置包含三个主要参数
```
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[
        bool,
        str,
        type[Exception],
        tuple[type[Exception], ...],
        Callable[[Exception], str],
    ]
```
- schema（必需参数）：与提供商策略的schema参数功能一致，支持 Pydantic模型 、TypedDict 、 JSON Schema 、 数据类(@dataclass) ，同时还支持联合类型 Union[类型1, 类型2] （允许模型根据输入内容选择最匹配的数据结构）。
- tool_message_content（可选参数）：用于自定义生成结构化输出时，会话历史中记录的提示信息。默认使用展示输出数据的标准响应语句。
- handle_errors（可选参数）：用于指定数据校验失败时的重试策略， 默认值为True 。

In [27]:
import os
from dataclasses import dataclass
from typing import Literal, TypedDict, Optional, Union

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError, \
    StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from rich import print as rprint
from typing_extensions import Annotated

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}}
)

## 1、schema参数
### 1.1 输出模式1-Pydantic类型

Pydantic类型的Schema支持数据验证，是优先推荐使用的方式。

In [ ]:
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 张三

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累积消费
    """
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (Str): 客户名称

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向{customer}发送感谢邮件"

class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_title: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户", description="客户等级")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="""
        请分析指定客户的情况：
        1. 先搜索客户数据库了解最新情况
        2. 如果是VIP客户，则发送感谢邮件
        3. 基于搜索结果生成结构化分析报告
        4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件
        """
    ),
    tools = (search_customer_database, send_email),
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
})

rprint(response)

if "structured_response" in response:
    structured_output = response["structured_response"]
    print(structured_output)

注意：
1. 如果是结构化输出，在系统提示词中最后提示结构化输出结果，如果提示词中先结构化输出结果（Agent已经执行完成），可能会导致一些工具不会再被调用。
2. 系统提示词中最后加入“未找到用户”时的处理提示，避免程序一直调用工具尝试查找对应用户信息。

### 1.2 输出模式2-TypedDict类型
TypedDict 允许为字典对象定义固定的键名和对应的 值类型 ，是 带有类型提示的字典结构 。具体的：
1. TypedDict字段定义采用 Annotated[类型, 默认值, "描述"]格式
2. 可选字段使用 Optional包装，默认值在 Annotated 中指定
3. TypedDict 不支持运行时验证。

In [ ]:
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 张三

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累积消费
    """
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (Str): 客户名称

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向{customer}发送感谢邮件"

class CustomerAnalysis(TypedDict):
    """客户分析报告"""
    customer_name: Annotated[Optional[str], None, "客户姓名"]
    customer_title: Annotated[Literal["潜在客户", "普通客户", "VIP客户", "流失风险"], "潜在客户", "客户等级"]
    recent_activity: Annotated[Optional[str], None, "最近活动"]
    spending_level:  Annotated[Literal["低", "中", "高"], None, "消费水平"]
    send_email: Annotated[bool, False, "是否已发送感谢邮件"]

agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="""
        请分析指定客户的情况：
        1. 先搜索客户数据库了解最新情况
        2. 如果是VIP客户，则发送感谢邮件
        3. 基于搜索结果生成结构化分析报告
        4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件
        """
    ),
    tools = (search_customer_database, send_email),
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
})

rprint(response)

if "structured_response" in response:
    structured_output = response["structured_response"]
    print(structured_output)

### 1.3 输出模式3-JsonSchema类型
JSON Schema是提供一个标准的 JSON Schema 字典来定义结构。适合需要与多种编程语言交互或进行复杂数据约束定义的场景。

In [ ]:
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 张三

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累积消费
    """
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (Str): 客户名称

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向{customer}发送感谢邮件"

customer_analysis_schema = {
    "title": "CustomerAnalysis",
    "type": "object",
    "description": "客户分析报告",
    "properties": {
    "customer_name": {
            "type": "string",
            "default": "",
            "description": "客户姓名"
        },
        "customer_tier": {
            "type": "string",
            "enum": ["潜在客户", "普通客户", "VIP客户", "流失风险"],
            "default": "潜在客户",
            "description": "客户等级"
        },
        "recent_activity": {
            "type": "string",
            "default": "",
            "description": "最近活动"
        },
        "spending_level": {
            "type": "string",
            "enum": ["低", "中", "高"],
            "default": "低",
            "description": "消费水平"
        },
        "send_email": {
            "type": "boolean",
            "default": False,
            "description": "是否已发送感谢邮件"
        }
    },
    # 所有字段都是必须输出的
    "required": ["customer_name", "customer_tier", "recent_activity", "spending_level"]
}

agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="""
        请分析指定客户的情况：
        1. 先搜索客户数据库了解最新情况
        2. 如果是VIP客户，则发送感谢邮件
        3. 基于搜索结果生成结构化分析报告
        4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件
        """
    ),
    tools = (search_customer_database, send_email),
    response_format=ToolStrategy(schema=customer_analysis_schema)
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
})

rprint(response)

if "structured_response" in response:
    structured_output = response["structured_response"]
    print(structured_output)

注意以上代码中定义json_schema的时候指定的title, description, type, properties, required是遵循JSON Schema 规范的标准关键字，是固定写法。

### 1.4 输出模式4-@dataclass类型
@dataclass是Python3.7引入的一个装饰器，用于简化数据存储类的定义。

In [ ]:
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 张三

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累积消费
    """
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (Str): 客户名称

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向{customer}发送感谢邮件"

@dataclass
class CustomerAnalysis:
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_title: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户", description="客户等级")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="""
        请分析指定客户的情况：
        1. 先搜索客户数据库了解最新情况
        2. 如果是VIP客户，则发送感谢邮件
        3. 基于搜索结果生成结构化分析报告
        4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件
        """
    ),
    tools = (search_customer_database, send_email),
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
})

rprint(response)

if "structured_response" in response:
    structured_output = response["structured_response"]
    print(structured_output)

### 1.5 多schema联合模式
ToolStrategy允许指定多个类型“ Union[类型1, 类型2] ”这种写法，LLM能够根据输入文本的内容，智能地选择 最合适的一个 数据模型（Schema）来生成结构化输出，但是最终会只有一种类型输出。

适用于根据不同输入内容，生成不同的结构化输出的场景，但是底层工具转换结构化输出只会转换成一种结构化类型输出。

In [ ]:
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(Union[ContactInfo, EventInfo])
)

response = agent.invoke(
{
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
}
)

for msg in response["messages"]:
    msg.pretty_print()
print(response["structured_response"])

## 2、tool_message_content参数-自定义工具消息
如果采用ToolStrategy策略处理结构化输出时，LangChain会在消息列表末尾追加一条Tool_message，让整个链路完整。但实际上没有实际的工具执行，这是一条伪消息。

我们可以通过ToolStrategy的 tool_message_content 参数定制其消息内容，将指定的内容写入对话历史的提示信息，这样做的好处如下：
1. 在最终用户可见的对话流中，使用 更自然的消息 替代原始数据。
2. 用简短的确认信息替代可能很长的数据块， 减少token消耗 。

当不设置 tool_message_content时，模型收到的 ToolMessage里就包含了像 {'name': '张三', 'email':'zhangsan@email.com'... ...} 这样的具体数据。当设置了tool_message_content时，模型收到的
ToolMessage只是一个预定义的确认信息，如“ 格式化输出成功！ ”。这种方式节省了上下文窗口的令牌消耗，并且让对话流对最终用户更友好。

说明： 无论 tool_message_content如何设置，成功提取的结构化数据最终都会正确存入result["structured_response"] 返回，自定义消息仅影响对话历史中的一条记录。

In [ ]:
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo, tool_message_content="已成功提取信息")
)

response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
    ]
})
rprint(response)

## 3、handle_errors参数-错误处理
受限于模型能力，大模型输出的内容可能并不符合格式要求，ToolStrategy通过其handle_errors参数提供了结构化过程错误处理策略，以下是主要的几种方式及其用途：
- handle_errors=True：LangChain默认方式，捕获所有异常，并使用LangChain内置的、信息明确的错误消息模版提示模型重试，确保最终能得到符合预定格式的有效数据。适用于大多数希望自动处理错误的通用场景。
- handle_errors=False：关闭自动重试机制，任何异常都会直接抛出，会中断程序运行。
- handle_errors="自定义字符串"：捕获所有异常，但使用开发者预设的固定字符串作为错误消息。适用于需要统一、友好的用户提示，或进行特定业务引导的场景。
- handle_errors=ExceptionType：仅捕获指定类型（如ValueError）或元组中的异常类型并进行重试，其他异常直接抛出。适用于需要精准控制，只对特定错误进行重试的场景。
- handle_errors=callable：灵活性最高的方式，使用开发者自定义的函数来处理异常，可根据不同的异常类型返回差异化的提示信息。适用于需要复杂、精细化错误处理的场景。

### 3.1 设置为True/False/固定字符串
设计思路：模型对于单条信息的格式化输出请求，输出了多个工具调用请求。也称为多结构化输出错误。

In [ ]:
class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成",
        handle_errors="请检查输入数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})
rprint(result)

1）设置为True，运行结果报错

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.\n Please fix your mistakes.

模型接收到错误反馈后，再次生成新的调用请求，直至成功或达到内部的最大重试次数。

ToolStrategy中自定义的tool_message_content控制的是成功后的消息内容，不影响错误消息内容。

2）设置为False，程序报错

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
During task with name 'model' and id 'e6cc4c27-292c-5f3d-cad2-cfe0ae63f799'

3）设置为“请检查输入数据”

以上代码中注意如下几点：
1. ToolStrategy允许指定多个类型“Union[ContactInfo, EventDetails]”这种写法，但是最终只会转换成一种结构化类型输出。
2. 当ToolStrategy通过“Union[ContactInfo, EventDetails]”指定多个类型时，在内部调用生成结构化类型工具会报错。此时：
   - handle_errors=True（默认值）开始发挥作用，系统会生成一个ToolMessage，明确告诉LLM“Error: Model incorrectly returned multiple structured responses (ContactInfo,EventDetails) when only one is expected.”，大模型收到这个精准的反馈后，会重新进行推理，最终选择并输出一个最符合要求的Schema。
   - 如果handle_errors 设置为False，执行代码过程直接报错。
3. 当格式化输出有错误时，Agent内部会进行工具调用重试，直到符合要求格式化输出前， 可能会进行多次重试 。

### 3.2 设置为指定异常类型
默认情况下，LangChain会处理结构化输出处理时抛出的两类异常：

1、MultipleStructuredOutputsError
多结构化输出错误，当返回的工具调用请求数量大于1时，抛出该异常，默认情况下LangChain会拦截异常并提醒模型重试。

2、StructuredOutputValidationError
输出结构化验证错误，当输出格式不符合结构化要求时，抛出上述异常。

默认情况下LangChain会拦截该异常并自动重试。

In [ ]:
class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成",
        handle_errors=(MultipleStructuredOutputsError, StructuredOutputValidationError)
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})
rprint(result)

### 3.3 设置为自定义错误处理函数

指定异常处理函数并返回字符串时，LangChain会在遇到异常时自动重试并将异常处理函数的返回值作为ToolMessage的内容。

我们也可以选择在异常处理函数中抛出异常。

In [ ]:
def custom_error_handler(error: Exception) -> str:
    """自定义错误处理器"""
    error_str = str(error)
    print(f"捕获到错误类型：{type(error).__name__}")
    print(f"错误详情：{error_str}")

    if isinstance(error, MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个进行返回"
    elif isinstance(error, StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符合要求"
    else:
        return f"Error: {error_str}"

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema=Union[ContactInfo, EventDetails],
        tool_message_content="提取完成",
        handle_errors=custom_error_handler
    )
)

result = agent.invoke({
    "messages": [{
    "role": "user",
    "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})
rprint(result)